# 02 · DeepEval metrics — what each one actually catches

**This notebook skips cleanly, cell by cell, if `deepeval` isn't installed or no judge API key is set — it never raises a raw traceback for a missing dependency.**

It wires four DeepEval LLM-as-judge metrics onto the 20-question benchmark:
**Faithfulness**, **Answer Relevancy**, **Contextual Precision**, and
**Contextual Recall**. Each metric is a gate — it carries a pass/fail
threshold, not just a bare number — so before any metric is run, this
notebook proves the pass/fail gate logic itself is correct on both a passing
and a failing case. Only then does it run the four metrics on one worked
example and explain, concretely, what failure each one is built to catch —
including cases where they *don't* catch the failure you'd expect.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `deepeval_status` | Reports whether `deepeval` is installed and a judge API key is set, without raising | `deepeval_status()` → `(False, "deepeval not installed...")` |
| `passes_gate` | The pass/fail comparison every DeepEval metric's `threshold` performs internally | `passes_gate(0.85, 0.70)` → `True` |
| `run_faithfulness` | Runs DeepEval's `FaithfulnessMetric`, or reports a graceful skip | `run_faithfulness(query, answer, retrieved_context)` |
| `run_answer_relevancy` | Runs DeepEval's `AnswerRelevancyMetric`, or reports a graceful skip | `run_answer_relevancy(query, answer, retrieved_context)` |
| `run_contextual_precision` | Runs DeepEval's `ContextualPrecisionMetric`, or reports a graceful skip | `run_contextual_precision(query, answer, retrieved_context, expected_output)` |
| `run_contextual_recall` | Runs DeepEval's `ContextualRecallMetric`, or reports a graceful skip | `run_contextual_recall(query, answer, retrieved_context, expected_output)` |


## Step 1 — locate the repo root and confirm the environment

This notebook lives two levels below the repo root, so the first thing it does is walk up the directory tree to find `nbio.py` and import it — everything else in this notebook depends on `repo_root` being set correctly.

In [ ]:
# This notebook lives two levels below the repo root (01-modules/06-bench/),
# and Jupyter starts a kernel with its working directory set to the
# notebook's own folder -- so nbio.py (at the repo root) is not importable
# yet. Walk up until we find it, same logic nbio.bootstrap() uses
# internally once it CAN be imported.
import sys
from pathlib import Path

def _find_repo_root(start):
    root = start.resolve()
    for _ in range(6):
        if (root / "nbio.py").is_file():
            return root
        root = root.parent
    raise RuntimeError("could not locate nbio.py above the current directory")

_repo_root_for_import = _find_repo_root(Path.cwd())
if str(_repo_root_for_import) not in sys.path:
    sys.path.insert(0, str(_repo_root_for_import))

import nbio
repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 2 — check whether `deepeval` and a judge-model key are available

This mirrors the check `eval.py` does — never assume a key or a package is
there; report which case you're in before anything downstream depends on
it.

In [ ]:
import os

def deepeval_status():
    try:
        import deepeval  # noqa: F401
    except ImportError:
        return False, "deepeval not installed (pip install -r 01-modules/06-bench/requirements.txt)"
    if not (os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")):
        return False, "no judge model API key set (OPENAI_API_KEY or GROQ_API_KEY)"
    return True, "available"

In [ ]:
available, reason = deepeval_status()
print("deepeval available:", available)
print("reason:", reason)

## Step 3 — the pass/fail gate every metric below relies on

Every DeepEval metric carries a `threshold` — a score at or above it passes,
below it fails. That comparison is the actual gate this notebook is
building toward, so it gets proven correct on its own, with plain numbers,
before any real judge call happens. Nothing below this cell assumes the gate
works until it's been shown to.

In [ ]:
def passes_gate(score, threshold):
    """The same score >= threshold comparison a DeepEval metric performs
    internally to set its own .success flag -- pulled out here so it can be
    checked on its own, with plain numbers, before any judge call happens."""
    return score is not None and score >= threshold

In [ ]:
pass_case = passes_gate(0.85, 0.70)
fail_case = passes_gate(0.40, 0.70)
print(f"pass case: passes_gate(0.85, 0.70) -> {pass_case}")
print(f"fail case: passes_gate(0.40, 0.70) -> {fail_case}")
assert pass_case is True
assert fail_case is False
print("\ngate confirmed correct on both a passing and a failing score")

## Step 4 — the worked example this notebook scores

One question, one retrieved context (deliberately incomplete — it's missing
the second ground-truth fact), and one answer (deliberately including one
claim the context does not support). This is constructed to make all four
metrics disagree with each other, which is the point: a single "answer
quality" number would hide exactly the information these four are meant to
separate out.

In [ ]:
query = "What is the recommended timing for skin grafting in deep partial thickness burns?"

# Only ONE of the two real ground-truth facts is present in this context --
# the split-thickness graft fact is missing. This is deliberate.
retrieved_context = [
    "Early tangential excision within 48-72 hours is associated with reduced "
    "blood loss and shorter hospital stay in deep partial thickness burns."
]

ground_truth_facts = [
    "Early tangential excision within 48-72 hours is associated with reduced blood loss and shorter hospital stay",
    "Split-thickness skin graft is the standard coverage for excised burn wounds",
]

# The answer restates the supported claim correctly, but ALSO adds a specific
# number (7-14 days) that appears nowhere in retrieved_context -- an
# unsupported claim injected into an otherwise-grounded answer. This is the
# shape of a real production failure mode, just injected deliberately here
# instead of arising from zero retrieval.
answer = (
    "Early tangential excision within 48 to 72 hours reduces blood loss and "
    "shortens hospital stay. Skin grafting should typically be completed "
    "within 7 to 14 days of the burn for best cosmetic outcomes."
)

print("query:", query)
print("retrieved_context:", retrieved_context)
print("answer:", answer)

## Step 5 — one shared spend ceiling for all four metrics

Each metric below calls a judge model (`DEEPEVAL_JUDGE_MODEL`, default `gpt-4o`) through DeepEval, which prices its own call internally as `metric.evaluation_cost`. One `nbio.cost_meter` is created here and passed into all four `run_*` functions, so the ceiling covers the notebook's real total spend, not just one metric's call.

In [ ]:
meter = nbio.Meter(budget_usd=0.50)  # spans all four metric cells below,
# so a plain object is used directly rather than the with-block form --
# no single "with" can wrap four separate cells
print("spend ceiling for this notebook: $0.50 (shared across all four metrics)")

## Step 6 — Faithfulness: does the answer follow from the retrieved context?

Catches: an answer that states something the context doesn't support — a
hallucinated fact, even inside an otherwise-correct answer. This is the
metric that would have caught the real failure this benchmark is named
after: an answer with specific numbers and zero supporting context scores
**low** faithfulness, because there is no context that could support it.

Does NOT catch: an answer that's faithful to bad context. If the retrieved
context itself is wrong, a perfectly faithful summary of it still scores
high faithfulness. Faithfulness measures agreement with what was retrieved,
not whether what was retrieved was correct.

In [ ]:
def run_faithfulness(query, answer, contexts, meter):
    available, reason = deepeval_status()
    if not available:
        return {"skipped": reason}
    from deepeval.metrics import FaithfulnessMetric
    from deepeval.test_case import LLMTestCase

    test_case = LLMTestCase(input=query, actual_output=answer, retrieval_context=contexts)
    metric = FaithfulnessMetric(threshold=0.70, model=os.getenv("DEEPEVAL_JUDGE_MODEL", "gpt-4o"))
    try:
        metric.measure(test_case)
        judge_model = str(getattr(metric, "evaluation_model", None) or metric.model)
        meter.record_cost(judge_model, getattr(metric, "evaluation_cost", 0) or 0)
        return {"score": round(float(metric.score), 3), "reason": getattr(metric, "reason", None)}
    except Exception as exc:  # noqa: BLE001
        return {"error": str(exc)}

In [ ]:
print(run_faithfulness(query, answer, retrieved_context, meter))

## Step 7 — Answer Relevancy: does the answer address the question asked?

Catches: an answer that's accurate but off-topic, padded, or answers a
different (even related) question. A retrieval system that returns strong
evidence about burn *classification* when asked about *grafting timing*
could still produce a low-relevancy answer even with perfect faithfulness.

Does NOT catch: hallucination. An invented but on-topic answer can score
high relevancy and low faithfulness at the same time — this is exactly why
the benchmark runs both metrics rather than either alone.

In [ ]:
def run_answer_relevancy(query, answer, contexts, meter):
    available, reason = deepeval_status()
    if not available:
        return {"skipped": reason}
    from deepeval.metrics import AnswerRelevancyMetric
    from deepeval.test_case import LLMTestCase

    test_case = LLMTestCase(input=query, actual_output=answer, retrieval_context=contexts)
    metric = AnswerRelevancyMetric(threshold=0.70, model=os.getenv("DEEPEVAL_JUDGE_MODEL", "gpt-4o"))
    try:
        metric.measure(test_case)
        judge_model = str(getattr(metric, "evaluation_model", None) or metric.model)
        meter.record_cost(judge_model, getattr(metric, "evaluation_cost", 0) or 0)
        return {"score": round(float(metric.score), 3), "reason": getattr(metric, "reason", None)}
    except Exception as exc:  # noqa: BLE001
        return {"error": str(exc)}

In [ ]:
print(run_answer_relevancy(query, answer, retrieved_context, meter))

## Step 8 — Contextual Precision: is the *relevant* retrieved content ranked ahead of the irrelevant?

Catches: a retriever that returns the right document buried behind five
wrong ones. This one needs `expected_output` (the ground truth) as well as
the context, because "precision" here means: of what was retrieved, is the
useful part ranked near the top?

Does NOT catch: retrieving nothing at all. An empty or single-item context
list doesn't have a meaningful "ranking" to score — this metric assumes
retrieval returned *something*, and is a poor tool for diagnosing a
zero-retrieval failure. That's what `paper_count` in `eval.py` is for.

In [ ]:
def run_contextual_precision(query, answer, contexts, expected_output, meter):
    available, reason = deepeval_status()
    if not available:
        return {"skipped": reason}
    from deepeval.metrics import ContextualPrecisionMetric
    from deepeval.test_case import LLMTestCase

    test_case = LLMTestCase(
        input=query, actual_output=answer, retrieval_context=contexts, expected_output=expected_output,
    )
    metric = ContextualPrecisionMetric(threshold=0.60, model=os.getenv("DEEPEVAL_JUDGE_MODEL", "gpt-4o"))
    try:
        metric.measure(test_case)
        judge_model = str(getattr(metric, "evaluation_model", None) or metric.model)
        meter.record_cost(judge_model, getattr(metric, "evaluation_cost", 0) or 0)
        return {"score": round(float(metric.score), 3), "reason": getattr(metric, "reason", None)}
    except Exception as exc:  # noqa: BLE001
        return {"error": str(exc)}

In [ ]:
expected_output = "\n".join(ground_truth_facts)
print(run_contextual_precision(query, answer, retrieved_context, expected_output, meter))

## Step 9 — Contextual Recall: did retrieval surface everything the ground truth needs?

Catches: exactly what this worked example was built to demonstrate — the
context here is missing the "split-thickness skin graft is standard
coverage" fact, so recall against the two `ground_truth_facts` should come
back showing that gap.

Does NOT catch: an answer that ignores context it was given. Recall scores
the *retrieved context* against the ground truth, not the *answer* against
the context — a generator that ignores perfect context still gets a high
recall score here, because the metric never looks at the answer text for
this part of the judgment.

In [ ]:
def run_contextual_recall(query, answer, contexts, expected_output, meter):
    available, reason = deepeval_status()
    if not available:
        return {"skipped": reason}
    from deepeval.metrics import ContextualRecallMetric
    from deepeval.test_case import LLMTestCase

    test_case = LLMTestCase(
        input=query, actual_output=answer, retrieval_context=contexts, expected_output=expected_output,
    )
    metric = ContextualRecallMetric(threshold=0.60, model=os.getenv("DEEPEVAL_JUDGE_MODEL", "gpt-4o"))
    try:
        metric.measure(test_case)
        judge_model = str(getattr(metric, "evaluation_model", None) or metric.model)
        meter.record_cost(judge_model, getattr(metric, "evaluation_cost", 0) or 0)
        return {"score": round(float(metric.score), 3), "reason": getattr(metric, "reason", None)}
    except Exception as exc:  # noqa: BLE001
        return {"error": str(exc)}

In [ ]:
print(run_contextual_recall(query, answer, retrieved_context, expected_output, meter))

print()
print(meter.report())

## Summary — one metric, one failure mode

| Metric | Catches | Blind to |
|---|---|---|
| Faithfulness | answer states something context doesn't support | context itself being wrong |
| Answer Relevancy | answer is off-topic or padded | hallucination that stays on-topic |
| Contextual Precision | relevant docs buried behind irrelevant ones | zero-retrieval (nothing to rank) |
| Contextual Recall | retrieval missed a needed fact | generator ignoring good context |

This is why `04-benchmarks/clinical-retrieval/LEADERBOARD.md` reports
`paper_count` and a citation check *alongside* these four — a zero-retrieval
failure needs a metric that looks at retrieval directly, not just at the
judge's opinion of the resulting text.